In [1]:
from dotenv import load_dotenv
import getpass
import os

if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your token:")

load_dotenv()

True

Data preparation

In [ ]:
import importlib
import eval_utils

importlib.reload(eval_utils)

In [2]:
from eval_utils import load_llm, load_embeddings, load_docs

# load chat model
chat_model = load_llm("hf_models/Llama-3.1-8B-Instruct")

# load embedding model
embedding = load_embeddings("sentence-transformers/all-MiniLM-L6-v2",
                            "hf_models/embeddings/all-MiniLM-L6-v2")

# load docs
docs = load_docs("data/mk4s_manuel")


/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.38it/s]
Device set to use cuda:0
/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/RAGAS/eval_utils.py:34: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(
100%|██████████| 8/8 [00:00<00:00, 10423.87it/s]


Chunking

In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=128, 
                                               chunk_overlap=16,
                                               separators=["\n## ","\n### ","\n#### ", "\n\n", ".", "\n", " "],
                                               is_separator_regex=False,
                                               keep_separator=False,)
chunks = text_splitter.split_documents(docs)

Initialize vector storage and test ingesting time

In [8]:
import time
import shutil
import pandas as pd
from langchain_community.vectorstores import FAISS, Chroma

times = {}

# FAISS ingestion timing
t0 = time.perf_counter()
vector_storage_faiss = FAISS.from_documents(chunks, embedding)
t1 = time.perf_counter()
times["FAISS"] = t1 - t0

retriever_faiss = vector_storage_faiss.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)


# Chroma ingestion timing
persist_directory = "./chroma_store_minilm"  
shutil.rmtree("chroma_store_minilm", ignore_errors=True)


vector_storage_chroma = Chroma.from_documents(
    documents=chunks, 
    embedding=embedding, 
    collection_name="my_collection_minilm", 
    persist_directory= persist_directory )

t2 = time.perf_counter()
vector_storage_chroma= Chroma.from_documents(chunks,embedding)
t3 = time.perf_counter()
times["Chroma"] = t3 - t2

retriever_chroma = vector_storage_chroma.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)


In [ ]:
# Create DataFrame 2×1
df_ingest_time = pd.DataFrame.from_dict(times, orient="index", columns=["ingest_seconds"])
print(df_ingest_time)

# Save Results
df_ingest_time.to_csv('evaluate_results/db_test/ingest_time.csv')

        ingest_seconds
FAISS         0.190086
Chroma        0.494329


Build Chain

In [ ]:
# build chain
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough,RunnableLambda
from langchain.schema.output_parser import StrOutputParser

template = """You are a helpful assistant specializing in 3D printing operations.  
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know. 
Use two sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

rag_chain_faiss = (
    {"context": retriever_faiss | RunnableLambda(format_docs) ,  
     "question": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser()
)

rag_chain_chroma = (
    {"context": retriever_faiss | RunnableLambda(format_docs),  
     "question": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser()
)

Generate evalutaion test dataset by RAGAS

In [11]:
# evaluation dataset preparasion
import pandas as pd
import time
from ragas import EvaluationDataset

def get_eval_ds(path, type:str):

    if type not in ['faiss', 'chroma']:
        raise ValueError("type is not valid, only input'faiss' or 'chroma'")
    
    if type == 'faiss':
        retriever = retriever_faiss
        rag_chain = rag_chain_faiss
    else:
        retriever = retriever_chroma
        rag_chain = rag_chain_chroma

    testset = pd.read_csv(path)
    queries = testset["user_input"].to_list()
    references = testset["reference"].to_list()

    # generate evaluation dataset
    rt_times = []
    dataset = []

    for query, reference in zip(queries, references):
        relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
        t0 = time.perf_counter()
        response = rag_chain.invoke(query)
        t1 = time.perf_counter()
        dataset.append(
            {
                "user_input": query,
                "retrieved_contexts": relevant_docs,
                "response": response,
                "reference": reference,
            }
        )
        rt_times.append(t1 -t0)

    eval_ds = EvaluationDataset.from_list(dataset)

    return eval_ds, rt_times

Using LLM to evaluate dataset

In [12]:
from ragas import evaluate
from ragas.metrics import LLMContextPrecisionWithReference, LLMContextRecall, AnswerRelevancy, ContextRelevance
from eval_utils import load_eval_model

def get_eval_result(eval_ds, metrics):
    evaluator_llm = load_eval_model()

    result = evaluate(
        dataset=eval_ds,
        metrics=metrics,
        llm=evaluator_llm,
        )
    
    return result


In [13]:
# select metrics
metrics=[LLMContextPrecisionWithReference(),
         LLMContextRecall(), 
         AnswerRelevancy(),
         ContextRelevance(),]



eval_ds_faiss, retrieval_time_faiss = get_eval_ds("evaluate_dataset/test_dataset.csv",'faiss')
eval_ds_chroma, retrieval_time_chroma = get_eval_ds("evaluate_dataset/test_dataset.csv",'chroma')

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [14]:
result_faiss = get_eval_result(eval_ds_faiss, metrics)
result_chroma = get_eval_result(eval_ds_chroma,metrics)

Evaluating:   1%|          | 1/160 [00:05<13:17,  5.02s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:   8%|▊         | 13/160 [00:20<02:53,  1.18s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  53%|█████▎    | 85/160 [02:42<02:14,  1.80s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  66%|██████▌   | 105/160 [03:22<01:54,  2.09s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:   1%|▏         | 2/160 [00:04<05:40,  2.15s/it]LLM returned 1 generations instead of request

In [15]:
print(result_faiss)
print(result_chroma)

{'llm_context_precision_with_reference': 0.5035, 'context_recall': 0.3508, 'answer_relevancy': 0.8717, 'nv_context_relevance': 0.7750}
{'llm_context_precision_with_reference': 0.4644, 'context_recall': 0.2671, 'answer_relevancy': 0.8968, 'nv_context_relevance': 0.6625}


In [ ]:
df_results_faiss = result_faiss.to_pandas()
df_results_faiss["retrieval_seconds"] = retrieval_time_faiss


df_results_chroma = result_chroma.to_pandas()
df_results_chroma["retrieval_seconds"] = retrieval_time_chroma

# save results
df_results_faiss.to_csv('evaluate_results/02_db_test/faiss.csv')
df_results_chroma.to_csv('evaluate_results/02_db_test/chroma.csv')